# Photo Restoration with Open-Source Diffusion Models
### Image-to-Image (`img2img`) + Inpainting with 🤗 `diffusers` — an open-source replacement for the OpenAI Image API

**Module context:** Specialist Diploma in Applied AI — Generative AI / Deep Learning
**Runtime required:** Colab → `Runtime ▸ Change runtime type ▸ T4 GPU` (free tier is enough)
**Cost:** $0. No API key, no account, no per-image billing.

---

## Why this notebook exists

The companion notebook `OpenAI_ImageRestort.ipynb` restores a damaged photograph by
sending it to a **hosted, closed-weights** model:

```python
response = client.responses.create(
    model="gpt-4.1",
    input=[{"role": "user", "content": [
        {"type": "input_text",  "text": prompt},
        {"type": "input_image", "image_url": f"data:image/jpeg;base64,{b64}"}]}],
    tools=[{"type": "image_generation"}],
)
```

That is one opaque call. You give it a prompt and a picture; a black box returns a picture.
You cannot see or control *how much* of the original survives, you cannot seed it for
reproducibility, and you cannot run it offline.

This notebook rebuilds the **same task** — take a damaged photo, return a clean one — on
**open weights you download and run yourself**, following the
[🤗 Diffusers image-to-image guide](https://huggingface.co/docs/diffusers/en/using-diffusers/img2img).

| | OpenAI Image / Responses API | 🤗 `diffusers` img2img |
|---|---|---|
| Weights | Closed, hosted | Open, downloaded to your GPU |
| Cost | Per image | Free after download |
| Works offline | No | Yes |
| Reproducible | No seed control | `torch.Generator().manual_seed(...)` |
| Control over "how much to change" | Prompt wording only | `strength` (a real number, 0→1) |
| Control over prompt adherence | None | `guidance_scale` |
| Say what to avoid | Awkward, in-prompt | `negative_prompt` |
| Repair only part of the image | Whole-image regeneration | `mask_image` (inpainting) |
| Swap the model | No | Any of ~thousands of checkpoints |
| Data leaves your machine | Yes | No |

---

## Learning outcomes

By the end of this notebook you will be able to:

1. Load an open diffusion checkpoint with `AutoPipelineForImage2Image` and run it on a photo.
2. Explain and demonstrate the effect of **`strength`**, **`guidance_scale`**, **`negative_prompt`** and **seed**.
3. Explain why `num_inference_steps` and `strength` *interact*, and compute the real step count.
4. Use **`AutoPipelineForInpainting`** with a mask to repair only the torn region of a photo.
5. **Chain** pipelines — SD 1.5 for the repair, SDXL Refiner for the final high-resolution pass.
6. Measure restoration quality with PSNR / SSIM and explain **why the numbers can get *worse* even when the picture looks better**.
7. Articulate the responsible-AI limits of generative restoration.

---

## The pipeline you are about to build

```
 ground-truth photo
        │  (we synthetically damage it, so we have an answer key)
        ▼
 damaged photo ──► Stage 1: SD 1.5 inpainting  ──► tear filled in
                        (mask = torn region)
                            │
                            ▼
                   Stage 2: SD 1.5 img2img     ──► creases / grain / sepia cleaned
                        (low strength, whole image)
                            │
                            ▼
                   Stage 3: SDXL Refiner img2img ──► 1024px, sharpened
                        (very low strength)
                            │
                            ▼
                      restored photo  ──► PSNR / SSIM vs ground truth
```

---
# Part 0 — Environment

## 0.1 Confirm you actually have a GPU

If the next cell errors or prints nothing, you are on a CPU runtime and every cell below will
take *minutes* instead of *seconds*. Fix it with `Runtime ▸ Change runtime type ▸ T4 GPU`.

In [ ]:
!nvidia-smi --query-gpu=name,memory.total,memory.free --format=csv

## 0.2 Install

Colab already ships PyTorch, Pillow, NumPy and scikit-image, so we only add the
Hugging Face stack. `-q` keeps the log short.

> **Note on versions.** This notebook was written against `diffusers==0.40.x`.
> Since `diffusers` 0.35 the loader argument is spelled **`dtype=`**, but the older
> **`torch_dtype=`** is still accepted and is what you will see in most tutorials —
> we use `torch_dtype=` so the code also runs on older pins.

In [ ]:
!pip install -q --upgrade "diffusers>=0.31" transformers accelerate safetensors
import diffusers, torch
print("diffusers:", diffusers.__version__)
print("torch    :", torch.__version__, "| CUDA:", torch.cuda.is_available())

## 0.3 Imports and shared helpers

Two helpers are worth reading carefully:

* **`resize_for_sd`** — Stable Diffusion's VAE downsamples by 8, so **every side of your
  image must be a multiple of 8**. Passing 513×769 will either crash or silently crop.
  This helper scales to a target long side and rounds both sides to the nearest multiple of 8.
* **`free_memory`** — a free T4 has ~15 GB. SD 1.5 (~2 GB), SD 1.5-inpainting (~2 GB) and the
  SDXL Refiner (~5 GB) will **not** all fit at once. You must `del` the old pipeline and then
  call this **before** loading the next one. This is the single most common cause of
  `CUDA out of memory` in student notebooks.

In [ ]:
import gc
import numpy as np
import torch
from PIL import Image, ImageDraw, ImageFilter

from diffusers import AutoPipelineForImage2Image, AutoPipelineForInpainting
from diffusers.utils import load_image, make_image_grid

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
DTYPE  = torch.float16 if DEVICE == "cuda" else torch.float32
print(f"device={DEVICE}  dtype={DTYPE}")


def free_memory():
    '''Release cached GPU memory. `del pipe` FIRST, then call this.'''
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
        torch.cuda.ipc_collect()
        free_gb, total_gb = (x / 1024**3 for x in torch.cuda.mem_get_info())
        print(f"GPU free: {free_gb:.1f} / {total_gb:.1f} GB")


def resize_for_sd(img, long_side=640, multiple=8):
    '''Resize keeping aspect ratio; force both sides to a multiple of 8.'''
    w, h = img.size
    scale = long_side / max(w, h)
    nw = max(multiple, int(round(w * scale / multiple)) * multiple)
    nh = max(multiple, int(round(h * scale / multiple)) * multiple)
    return img.resize((nw, nh), Image.LANCZOS)


def label(img, text):
    '''Burn a caption onto a copy of the image so grids are self-explanatory.'''
    img = img.convert("RGB").copy()
    d = ImageDraw.Draw(img)
    d.rectangle([0, 0, img.width, 20], fill=(0, 0, 0))
    d.text((5, 5), text, fill=(255, 255, 255))
    return img


def grid(images, captions=None, cols=None):
    '''make_image_grid requires equal sizes -- normalise, caption, then tile.'''
    w, h = images[0].size
    images = [im.convert("RGB").resize((w, h), Image.LANCZOS) for im in images]
    if captions:
        images = [label(im, c) for im, c in zip(images, captions)]
    cols = cols or len(images)
    rows = (len(images) + cols - 1) // cols
    while len(images) < rows * cols:                  # pad so the grid is rectangular
        images.append(Image.new("RGB", (w, h), (30, 30, 30)))
    return make_image_grid(images, rows=rows, cols=cols)


def seeded(seed):
    '''A generator pinned to the compute device -- this is what makes runs repeatable.'''
    return torch.Generator(device=DEVICE).manual_seed(seed)


def enable_vae_savers(pipe):
    '''Turn on sliced VAE decoding, whichever diffusers version you are on.

    diffusers <= 0.35 : pipe.enable_vae_slicing()
    diffusers >= 0.36 : the method was REMOVED from the pipeline and now lives
                        on the VAE itself -> pipe.vae.enable_slicing()

    Calling the old name on a recent version raises
    AttributeError: 'StableDiffusionXLImg2ImgPipeline' object has no attribute
    'enable_vae_slicing'.  Feature-detect instead of guessing.
    '''
    if hasattr(pipe, "vae") and hasattr(pipe.vae, "enable_slicing"):
        pipe.vae.enable_slicing()
        print("VAE slicing on (pipe.vae.enable_slicing)")
    elif hasattr(pipe, "enable_vae_slicing"):
        pipe.enable_vae_slicing()
        print("VAE slicing on (legacy pipe.enable_vae_slicing)")
    else:
        print("VAE slicing not available in this diffusers build -- continuing without it")

---
# Part 1 — Build a damaged photograph (with an answer key)

Real archival photos have no *ground truth*: you have the damaged scan and nothing else, so
you can only judge the restoration by eye. That is fine for a demo but useless for teaching
and impossible to grade.

So we do the opposite: take a **clean** photo, **damage it ourselves**, and keep the clean
version as the answer key. Now every restoration can be scored against a known target
(Part 6), and — more importantly — students can *see* exactly which artefacts the model
removed and which detail it **invented**.

The damage function stacks the five things that actually happen to old prints:

| Artefact | Physical cause | Our simulation |
|---|---|---|
| Sepia cast | Silver → silver sulphide over decades | sepia colour matrix |
| Faded contrast | Light / humidity bleaching the dyes | blend toward mid-grey |
| Fold lines | Photo kept folded in a wallet | long near-vertical bright lines |
| Scratches & dust | Abrasion, dirty scanner glass | many short random bright lines |
| Grain | Film emulsion + scanner noise | Gaussian noise |
| **A physical tear** | Print ripped and taped | a jagged band of paper-white **plus a mask** |

The tear is handled separately because it is **information that is genuinely gone**.
Nothing can recover it — Part 4 asks the model to *invent* something plausible there,
which is a very different operation from cleaning up noise.

In [ ]:
# ---------------------------------------------------------------- damage toolkit
def _np(img):
    return np.asarray(img.convert("RGB")).astype(np.float32)


def apply_sepia(img, strength=0.7):
    a = _np(img) / 255.0
    m = np.array([[0.393, 0.769, 0.189],
                  [0.349, 0.686, 0.168],
                  [0.272, 0.534, 0.131]], dtype=np.float32)
    sep = np.clip(a @ m.T, 0.0, 1.0)
    out = (1 - strength) * a + strength * sep
    return Image.fromarray((out * 255).astype(np.uint8))


def fade(img, amount=0.30):
    '''Wash the image toward a pale grey -- classic loss of contrast.'''
    a = _np(img) / 255.0
    out = a * (1 - amount) + amount * 0.78
    return Image.fromarray((np.clip(out, 0, 1) * 255).astype(np.uint8))


def add_grain(img, sigma=10.0, rng=None):
    rng = rng or np.random.default_rng(0)
    a = _np(img)
    return Image.fromarray(np.clip(a + rng.normal(0, sigma, a.shape), 0, 255).astype(np.uint8))


def add_creases(img, rng, n_folds=2, n_scratches=70):
    img = img.convert("RGB")
    w, h = img.size
    overlay = Image.new("L", (w, h), 0)
    d = ImageDraw.Draw(overlay)

    # long vertical fold lines
    for i in range(n_folds):
        x = int(w * (i + 1) / (n_folds + 1))
        d.line([(x + int(rng.integers(-5, 6)), 0),
                (x + int(rng.integers(-5, 6)), h)],
               fill=195, width=3)

    # short random scratches and dust
    for _ in range(n_scratches):
        x0, y0 = int(rng.integers(0, w)), int(rng.integers(0, h))
        x1, y1 = x0 + int(rng.integers(-55, 56)), y0 + int(rng.integers(-55, 56))
        d.line([(x0, y0), (x1, y1)], fill=int(rng.integers(120, 225)),
               width=int(rng.integers(1, 3)))

    overlay = overlay.filter(ImageFilter.GaussianBlur(0.9))
    paper = Image.new("RGB", (w, h), (246, 241, 231))
    return Image.composite(paper, img, overlay)   # white where overlay is bright


def tear_out(img, rng, band_center=0.66, band_width=0.09):
    '''Rip a jagged vertical band out of the print.

    Returns (torn_image, mask) where the mask is WHITE (255) exactly on the
    missing paper -- that is the convention AutoPipelineForInpainting expects.
    '''
    img = img.convert("RGB")
    w, h = img.size
    jx, jh = max(1, int(0.030 * w)), max(1, int(0.012 * w))

    left, right = [], []
    for i in range(25):
        y = int(h * i / 24)
        c = band_center * w + int(rng.integers(-jx, jx + 1))
        half = band_width * w / 2 + int(rng.integers(-jh, jh + 1))
        left.append((int(c - half), y))
        right.append((int(c + half), y))

    mask = Image.new("L", (w, h), 0)
    ImageDraw.Draw(mask).polygon(left + right[::-1], fill=255)

    torn = Image.composite(Image.new("RGB", (w, h), (251, 248, 241)), img, mask)
    return torn, mask


def make_damaged(clean, seed=7):
    '''Full damage stack. Returns (damaged_photo, tear_mask).'''
    rng = np.random.default_rng(seed)
    x = apply_sepia(clean, 0.70)
    x = fade(x, 0.28)
    x = add_creases(x, rng, n_folds=2, n_scratches=70)
    x = add_grain(x, sigma=10.0, rng=rng)
    x, mask = tear_out(x, rng)
    return x, mask

print("damage toolkit ready")

## 1.1 Load a source photograph

We pull a stable Hugging Face documentation image so the notebook runs end-to-end with no
uploads. Cell 1.2 lets you swap in a real family photo instead — which is the version worth
doing in class.

In [ ]:
CANDIDATE_URLS = [
    "https://huggingface.co/datasets/huggingface/documentation-images/resolve/main/diffusers/inpaint.png",
    "https://huggingface.co/datasets/huggingface/documentation-images/resolve/main/diffusers/img2img-init.png",
]

clean = None
for url in CANDIDATE_URLS:
    try:
        clean = load_image(url)
        print("loaded:", url)
        break
    except Exception as e:
        print("failed:", url, "->", type(e).__name__)

if clean is None:
    raise RuntimeError("No source image could be downloaded. Use cell 1.2 to upload your own.")

clean = resize_for_sd(clean, long_side=640)
print("ground-truth size:", clean.size)
clean

## 1.2 *(Optional)* Use your own photo

Run this cell to upload a JPG/PNG from your machine. A genuinely old scan works best —
then skip the synthetic damage in 1.3 and set `damaged = clean` yourself, drawing your own
mask in 4.1.

In [ ]:
USE_MY_OWN_PHOTO = False   # <-- flip to True, then run this cell

if USE_MY_OWN_PHOTO:
    from google.colab import files
    uploaded = files.upload()
    path = next(iter(uploaded))
    clean = resize_for_sd(Image.open(path).convert("RGB"), long_side=640)
    print("using:", path, clean.size)
    display(clean)
else:
    print("Skipped -- still using the downloaded sample photo.")

## 1.3 Damage it

In [ ]:
damaged, tear_mask = make_damaged(clean, seed=7)

grid([clean, damaged, tear_mask.convert("RGB")],
     ["1. ground truth (answer key)", "2. damaged input", "3. tear mask (white = missing)"])

---
# Part 2 — The core replacement: `AutoPipelineForImage2Image`

This is the cell that replaces the entire OpenAI API call.

### What `img2img` actually does

Text-to-image starts from **pure noise** and denoises for `num_inference_steps`.
Image-to-image starts from **your photo**, pushes it partway back into noise, and denoises
from there. How far back it is pushed is `strength`:

```
strength = 0.0  ->  no noise added      ->  output ≈ input, model does nothing
strength = 0.3  ->  a little noise      ->  texture and colour change, composition intact
strength = 0.8  ->  almost pure noise   ->  only a loose echo of the original remains
strength = 1.0  ->  pure noise          ->  equivalent to text-to-image
```

**For restoration we want LOW strength (≈0.2–0.4).** The damage lives in the
high-frequency texture — grain, scratches, colour cast — and that is precisely what the
early denoising steps rewrite. Faces, poses and composition live in the low-frequency
structure, which low strength leaves alone. Turn the strength up and you stop restoring the
photo and start replacing it with a different one.

### The trap: `strength` silently changes your step count

`diffusers` runs only `int(num_inference_steps × strength)` steps. So:

| `num_inference_steps` | `strength` | steps actually run |
|---|---|---|
| 50 | 0.8 | 40 |
| 50 | 0.3 | 15 |
| 10 | 0.2 | **2** ← output will look broken |

When you lower `strength`, **raise `num_inference_steps`** to compensate. We use 50 below so
that even `strength=0.15` still gets 7 real steps.

In [ ]:
MODEL_SD15 = "stable-diffusion-v1-5/stable-diffusion-v1-5"

load_kwargs = dict(torch_dtype=DTYPE, use_safetensors=True)
if DEVICE == "cuda":
    load_kwargs["variant"] = "fp16"          # half-size download, GPU only

# SD 1.5 only: disable the NSFW checker AT LOAD TIME. It misfires on old portraits
# and returns an all-black image. Passing it here is safer than assigning
# `pipe.safety_checker = None` afterwards, which fights the config system on some
# versions. SDXL has no safety checker, so these two keys are NOT added to load_kwargs.
SD15_KWARGS = {**load_kwargs, "safety_checker": None, "requires_safety_checker": False}

pipe_i2i = AutoPipelineForImage2Image.from_pretrained(MODEL_SD15, **SD15_KWARGS)
pipe_i2i = pipe_i2i.to(DEVICE)
pipe_i2i.set_progress_bar_config(disable=True)

print("loaded:", MODEL_SD15)
free_memory()

## 2.1 The prompts

Two prompts, and the **negative** one does most of the work here.

* The **positive prompt** describes the *destination*: what a restored print looks like.
* The **negative prompt** describes the *artefacts to erase*. Every damage type from Part 1
  is named explicitly. This is the single biggest quality lever in restoration, and it has
  no equivalent in the OpenAI call — there you would have to write "please do not produce
  scratches" into the prompt and hope.

In [ ]:
RESTORE_PROMPT = (
    "a professionally restored vintage photograph, pristine undamaged print, "
    "clean even lighting, natural realistic colour, accurate skin tones, "
    "sharp focus, fine detail, high resolution archival scan"
)

NEGATIVE_PROMPT = (
    "scratch, scratches, crease, creases, fold line, tear, rip, torn paper, "
    "dust, speckles, stains, water damage, sepia, yellowed, faded, washed out, "
    "film grain, noise, blur, low resolution, jpeg artifacts, "
    "watermark, text, signature, deformed face, distorted anatomy, extra fingers"
)

print(f"positive: {len(RESTORE_PROMPT.split())} words")
print(f"negative: {len(NEGATIVE_PROMPT.split())} words")

## 2.2 First restoration — one line replaces the whole API call

Compare the signatures:

```python
# OpenAI  -- one opaque knob: the prompt
client.responses.create(model="gpt-4.1", input=[...], tools=[{"type": "image_generation"}])

# diffusers -- five explicit knobs, all inspectable
pipe_i2i(prompt=..., negative_prompt=..., image=..., strength=..., guidance_scale=..., generator=...)
```

In [ ]:
%%time
restored_v1 = pipe_i2i(
    prompt=RESTORE_PROMPT,
    negative_prompt=NEGATIVE_PROMPT,
    image=damaged,
    strength=0.35,
    guidance_scale=7.5,
    num_inference_steps=50,       # -> int(50 * 0.35) = 17 real steps
    generator=seeded(42),
).images[0]

grid([clean, damaged, restored_v1],
     ["ground truth", "damaged", "restored (s=0.35)"])

---
# Part 3 — Parameter lab

Three sweeps. Run them, then answer the questions underneath each one — they are the
assessment hooks.

## 3.1 `strength` — how much of the original survives

Everything else is held fixed, **including the seed**, so any difference you see is caused
by `strength` alone.

In [ ]:
%%time
STRENGTHS = [0.15, 0.25, 0.35, 0.50, 0.70]
outs, caps = [damaged], ["damaged input"]

for s in STRENGTHS:
    img = pipe_i2i(
        prompt=RESTORE_PROMPT,
        negative_prompt=NEGATIVE_PROMPT,
        image=damaged,
        strength=s,
        guidance_scale=7.5,
        num_inference_steps=50,
        generator=seeded(42),          # same seed every time -- isolate the variable
    ).images[0]
    outs.append(img)
    caps.append(f"strength={s}  ({int(50*s)} steps)")

grid(outs, caps, cols=3)

> **Q1.** At which `strength` do the fold lines disappear? At which `strength` does the
> *subject* start to change (different face, different object, different background)?
> The usable window is between those two numbers — write it down.
>
> **Q2.** At `strength=0.15` only 7 steps run. Is the output under-restored because the
> strength is too low, or because 7 steps is too few? Design a single experiment that
> separates the two causes. *(Hint: what happens at `strength=0.15, num_inference_steps=150`?)*

## 3.2 `guidance_scale` — how hard the model is pushed toward the prompt

Classifier-free guidance runs the model twice each step — once with your prompt, once with
the negative prompt — and extrapolates *away* from the negative. `guidance_scale` is how far
it extrapolates.

* **1.0** — guidance effectively off. The negative prompt is ignored, so the scratches stay.
* **6–9** — the normal working range.
* **15+** — over-saturated, plasticky, high-contrast. The model obeys the prompt so hard it
  stops respecting the photo.

In [ ]:
%%time
GUIDANCES = [1.5, 5.0, 7.5, 11.0, 16.0]
outs, caps = [damaged], ["damaged input"]

for g in GUIDANCES:
    img = pipe_i2i(
        prompt=RESTORE_PROMPT,
        negative_prompt=NEGATIVE_PROMPT,
        image=damaged,
        strength=0.35,
        guidance_scale=g,
        num_inference_steps=50,
        generator=seeded(42),
    ).images[0]
    outs.append(img)
    caps.append(f"guidance={g}")

grid(outs, caps, cols=3)

> **Q3.** `guidance_scale=1.5` leaves the damage largely intact. Explain why, in terms of
> what the negative prompt is doing in the classifier-free guidance equation.
>
> **Q4.** Compare `guidance=16.0` against `guidance=7.5`. Name two specific visual defects
> that over-guidance introduces.

## 3.3 Seed — the reproducibility the hosted API cannot give you

Same prompt, same image, same strength, same guidance — only the seed changes.
Every output below is a *different valid restoration*. This is the honest core of the
method: diffusion restoration is **sampling from a distribution of plausible originals**,
not recovering the one true original.

Note what this buys you in a teaching or forensic setting: because the seed is explicit,
any result here can be reproduced bit-for-bit by anyone, forever. The OpenAI call cannot
make that promise.

In [ ]:
%%time
SEEDS = [0, 42, 1234, 2026]
outs, caps = [], []

for s in SEEDS:
    img = pipe_i2i(
        prompt=RESTORE_PROMPT,
        negative_prompt=NEGATIVE_PROMPT,
        image=damaged,
        strength=0.45,                 # raised so the variation is visible
        guidance_scale=7.5,
        num_inference_steps=50,
        generator=seeded(s),
    ).images[0]
    outs.append(img)
    caps.append(f"seed={s}")

grid(outs, caps, cols=2)

> **Q5.** Pick one detail that differs across the four seeds. Would that detail be
> admissible as evidence about what the original photograph contained? Justify your answer.

---
# Part 4 — Inpainting the tear

`img2img` rewrites **every** pixel a little. That is the right tool for grain and colour
cast, and the wrong tool for the tear: the torn band contains no information at all, so
nudging it produces clean white paper rather than content.

**Inpainting** solves the right problem. You supply a **mask**, and the model regenerates
*only* the white region while conditioning on the surrounding pixels for context.

### Mask conventions — get these wrong and nothing works

* Mode **`L`** (8-bit greyscale), **same size** as the image.
* **White (255) = repaint this.** Black (0) = keep untouched. Inverting this is the single
  most common inpainting bug.
* **Dilate** the mask a few pixels. The tear has feathered edges with leftover paper-white
  fringe; if the mask stops exactly at the tear the fringe survives as a bright halo.
* **Blur** the mask. A hard edge makes the seam visible. `pipeline.mask_processor.blur()`
  feathers it so the repair blends.

In [ ]:
# free the img2img pipeline before loading the inpainting one -- T4 will not hold both
del pipe_i2i
free_memory()

MODEL_INPAINT = "stable-diffusion-v1-5/stable-diffusion-inpainting"

pipe_inpaint = AutoPipelineForInpainting.from_pretrained(MODEL_INPAINT, **SD15_KWARGS)
pipe_inpaint = pipe_inpaint.to(DEVICE)
pipe_inpaint.set_progress_bar_config(disable=True)

print("loaded:", MODEL_INPAINT)
free_memory()

## 4.1 Prepare the mask: dilate, then feather

In [ ]:
mask_dilated  = tear_mask.filter(ImageFilter.MaxFilter(9))        # grow ~4 px each side
mask_feather  = pipe_inpaint.mask_processor.blur(mask_dilated, blur_factor=12)

grid([tear_mask.convert("RGB"), mask_dilated.convert("RGB"), mask_feather.convert("RGB")],
     ["raw mask", "dilated (MaxFilter 9)", "feathered (blur 12)"])

## 4.2 Fill the tear

`strength=1.0` is correct **here and only here**. Inside the mask there is nothing worth
preserving, so we want a full regeneration. Outside the mask the pipeline composites the
original pixels straight back, so a high strength costs us nothing.

The prompt no longer describes restoration — it describes **what should be behind the tear**.
Change it and the model invents something else.

In [ ]:
%%time
INPAINT_PROMPT  = "continuation of the photograph, seamless, matching lighting and texture, natural, photorealistic"
INPAINT_NEG     = "white paper, blank, seam, edge, border, text, watermark, blurry, distorted"

filled = pipe_inpaint(
    prompt=INPAINT_PROMPT,
    negative_prompt=INPAINT_NEG,
    image=damaged,
    mask_image=mask_feather,
    strength=1.0,
    guidance_scale=7.5,
    num_inference_steps=50,
    generator=seeded(42),
).images[0]

filled = filled.resize(damaged.size, Image.LANCZOS)   # pipeline may pad to /8

grid([damaged, mask_feather.convert("RGB"), filled, clean],
     ["damaged (torn)", "mask", "tear filled", "ground truth"], cols=2)

> **Q6.** Compare the filled band against the ground truth. The model did not recover the
> original content — it produced *something plausible*. In one sentence, state the
> difference between **restoration** and **fabrication**, and say which one this is.
>
> **Q7.** Re-run 4.2 with `strength=0.6`. Explain the result using the masking mechanics
> described above.

---
# Part 5 — Chaining: SD 1.5 → SDXL Refiner

The 🤗 guide's **[chained pipelines](https://huggingface.co/docs/diffusers/en/using-diffusers/img2img#chained-image-to-image-pipelines)**
pattern: the output of one pipeline is the input of the next. Our chain is

```
damaged ──inpaint──► tear filled ──img2img (SD1.5)──► cleaned ──img2img (SDXL refiner @1024)──► final
```

**SDXL Refiner** (`stabilityai/stable-diffusion-xl-refiner-1.0`) was trained specifically on
the *last* denoising steps, so it is an unusually good final-polish img2img model. Two rules:

1. Feed it **≈1024 px**. It was trained there and degrades badly at 512.
2. Use **very low strength (0.2–0.3)**. It is a polish pass, not a rewrite.

### Memory
The refiner is ~5 GB in fp16. We use **`enable_model_cpu_offload()`**, which streams each
submodule to the GPU only while it runs and parks the rest in system RAM. It is slower than
`.to("cuda")` but it is what makes SDXL fit on a free T4.

> ⚠️ After `enable_model_cpu_offload()`, **never** call `.to("cuda")` on that pipeline —
> `accelerate` owns device placement and the two will fight.

> ⚠️ **API moved in diffusers 0.36.** Sliced VAE decoding used to be
> `pipeline.enable_vae_slicing()`. That method was **removed from the pipeline** and now
> lives on the VAE: **`pipeline.vae.enable_slicing()`**. Most tutorials online still show
> the old name, which now fails with
> `AttributeError: 'StableDiffusionXLImg2ImgPipeline' object has no attribute 'enable_vae_slicing'`.
> Our `enable_vae_savers()` helper feature-detects both, so this notebook runs on either.
> The same move applies to tiling: `pipeline.vae.enable_tiling()`.

In [ ]:
# Stage 2 of the chain: clean the creases/grain/sepia on the tear-filled image.
del pipe_inpaint
free_memory()

pipe_i2i = AutoPipelineForImage2Image.from_pretrained(MODEL_SD15, **SD15_KWARGS).to(DEVICE)
pipe_i2i.set_progress_bar_config(disable=True)

cleaned = pipe_i2i(
    prompt=RESTORE_PROMPT,
    negative_prompt=NEGATIVE_PROMPT,
    image=filled,
    strength=0.35,
    guidance_scale=7.5,
    num_inference_steps=50,
    generator=seeded(42),
).images[0]

grid([filled, cleaned], ["after inpainting", "after img2img clean-up"])

In [ ]:
%%time
# Stage 3: SDXL Refiner polish at 1024 px
del pipe_i2i
free_memory()

MODEL_REFINER = "stabilityai/stable-diffusion-xl-refiner-1.0"

pipe_refiner = AutoPipelineForImage2Image.from_pretrained(MODEL_REFINER, **load_kwargs)
pipe_refiner.enable_model_cpu_offload()      # NOT .to("cuda") -- see note above
enable_vae_savers(pipe_refiner)              # decode the VAE in slices, saves ~1 GB
pipe_refiner.set_progress_bar_config(disable=True)

cleaned_1024 = resize_for_sd(cleaned, long_side=1024)
print("refiner input:", cleaned_1024.size)

final = pipe_refiner(
    prompt=RESTORE_PROMPT,
    negative_prompt=NEGATIVE_PROMPT,
    image=cleaned_1024,
    strength=0.25,               # polish only
    guidance_scale=6.0,
    num_inference_steps=45,      # -> ~11 real steps
    generator=seeded(42),
).images[0]

print("final:", final.size)
final

In [ ]:
grid([clean, damaged, filled, cleaned, final],
     ["0. ground truth", "1. damaged", "2. +inpaint", "3. +img2img", "4. +SDXL refiner"],
     cols=3)

---
# Part 6 — Measure it (and learn to distrust the number)

Because we kept the answer key we can compute real metrics:

* **PSNR** (dB, higher better) — pure per-pixel error. Brutally literal.
* **SSIM** (0–1, higher better) — compares local luminance, contrast and structure. Closer
  to human judgement, but still a *similarity to the original* measure.

### The result you should expect — and must be able to explain

The restored image will often score **only slightly better, or even worse**, than the
damaged one, while looking dramatically better to your eye. Three reasons:

1. **Global colour correction is heavily penalised.** Removing the sepia cast shifts every
   pixel. PSNR counts each shift as error, even though it is the whole point.
2. **Invented detail is penalised.** The inpainted band is plausible but pixel-wise wrong,
   so it is scored as a large error region.
3. **Sharp ≠ correct.** New texture in the right *style* but the wrong *place* lowers PSNR
   while raising perceived quality.

This is the gap between **distortion** metrics (PSNR/SSIM) and **perceptual** metrics
(LPIPS, FID, NIQE). It is also exactly why "the model made my photo look better" is not
the same claim as "the model recovered my photo", which matters the moment anyone wants to
treat a restored image as a record of fact.

In [ ]:
from skimage.metrics import peak_signal_noise_ratio as psnr
from skimage.metrics import structural_similarity as ssim

def score(ref, test):
    '''PSNR / SSIM of `test` against reference `ref`, sizes normalised.'''
    test = test.convert("RGB").resize(ref.size, Image.LANCZOS)
    a = np.asarray(ref.convert("RGB"))
    b = np.asarray(test)
    return psnr(a, b, data_range=255), ssim(a, b, channel_axis=2, data_range=255)

rows = [
    ("damaged input",        damaged),
    ("stage 1: inpainted",   filled),
    ("stage 2: img2img",     cleaned),
    ("stage 3: SDXL refiner", final),
]

print(f"{'stage':<24}{'PSNR (dB)':>12}{'SSIM':>10}")
print("-" * 46)
for name, img in rows:
    p, s = score(clean, img)
    print(f"{name:<24}{p:>12.2f}{s:>10.4f}")

> **Q8.** Did the PSNR of the final image beat the damaged input? Whether it did or not,
> explain the number using the three reasons above.
>
> **Q9.** Propose a metric or protocol that *would* fairly reward this restoration. State
> one weakness of your proposal.

---
# Part 7 — Exercises

Work through these in order. Each has a `TODO` cell below it.

### Exercise 1 — Find the restoration sweet spot *(easy)*
Grid-search `strength ∈ {0.20, 0.30, 0.40}` × `guidance_scale ∈ {5, 7.5, 10}` (9 images, one
fixed seed). Score every result with `score(clean, img)` and print a 3×3 table of SSIM.
Report the best cell and say whether the highest SSIM is also the one you would pick by eye.

In [ ]:
# TODO Exercise 1
# for s in [0.20, 0.30, 0.40]:
#     for g in [5.0, 7.5, 10.0]:
#         ...

### Exercise 2 — Ablate the negative prompt *(easy)*
Run the same restoration three times: (a) `negative_prompt=None`, (b) a short negative
(`"scratches, noise"`), (c) the full `NEGATIVE_PROMPT`. Everything else fixed. Show the
three side by side and state which artefact each added term actually removed.

In [ ]:
# TODO Exercise 2

### Exercise 3 — Break the step/strength interaction *(medium)*
Hold `strength=0.2` and sweep `num_inference_steps ∈ {10, 25, 50, 150}`. Print the real step
count `int(steps*strength)` next to each image. At what real step count does quality stop
improving? This is your answer to **Q2**.

In [ ]:
# TODO Exercise 3

### Exercise 4 — Hand-authored mask *(medium)*
Damage a *different* region of `clean` (use `tear_out` with a different `band_center`, or
draw your own polygon with `ImageDraw`). Build the mask yourself, dilate and feather it, and
inpaint. Then deliberately **invert** the mask and run it again — explain the output.

In [ ]:
# TODO Exercise 4

### Exercise 5 — Swap the backbone *(harder)*
Replace `MODEL_SD15` with another img2img-capable checkpoint, e.g.
`"kandinsky-community/kandinsky-2-2-decoder"` or an SD 1.5 fine-tune from the Hub.
`AutoPipelineForImage2Image` handles the architecture difference for you — that is the
point of the Auto classes. Compare against SD 1.5 at identical `strength`, `guidance_scale`
and seed, and comment on whether "same seed" means anything across two different models.

In [ ]:
# TODO Exercise 5

### Exercise 6 — Real archival photo *(open-ended)*
Bring a genuine old scan (cell 1.2). There is no ground truth, so PSNR/SSIM are unavailable.
Define an evaluation protocol you could defend — e.g. blind A/B ranking by three classmates,
with the restored and damaged versions presented in random order — run it, and report the
result together with its limitations.

In [ ]:
# TODO Exercise 6

---
# Part 8 — Troubleshooting

| Symptom | Cause | Fix |
|---|---|---|
| `CUDA out of memory` | A previous pipeline is still resident | `del pipe` **then** `free_memory()`; add `enable_model_cpu_offload()` and `enable_vae_slicing()` |
| Output is a **black image** | The safety checker fired (common on portraits and old prints) | `pipe.safety_checker = None` (already set in this notebook) |
| Output ≈ identical to input | `strength` too low, or real step count `int(steps*strength)` is ~1–3 | Raise `strength` to ≥0.25 and `num_inference_steps` to ≥50 |
| Output is a *different photo* | `strength` too high | Drop to 0.2–0.4 |
| Damage survives the pass | `guidance_scale` too low, so the negative prompt is barely applied | Raise to 7–9 and name the artefact explicitly in `NEGATIVE_PROMPT` |
| Over-saturated, plastic look | `guidance_scale` too high | Drop to 6–8 |
| Inpainting repaints the whole image | Mask inverted | White = repaint. Check with `display(mask)` |
| Visible seam around the repair | Mask edge too hard | `MaxFilter` to dilate, then `mask_processor.blur(..., blur_factor=12+)` |
| `ValueError` about height/width | A side is not a multiple of 8 | Always pass images through `resize_for_sd` |
| SDXL output looks mushy | Fed at 512 px | Resize to ~1024 before the refiner |
| `.to("cuda")` errors after offload | `accelerate` owns placement | Pick one: `.to(DEVICE)` **or** `enable_model_cpu_offload()` |
| `AttributeError: ... has no attribute 'enable_vae_slicing'` | Removed from the pipeline in diffusers 0.36 | Use `pipe.vae.enable_slicing()` (and `pipe.vae.enable_tiling()`), or the `enable_vae_savers()` helper in 0.3 |
| `AttributeError: ... has no attribute 'enable_xformers_memory_efficient_attention'` | Also removed; PyTorch 2.x SDPA is the default and is already efficient | Delete the call |
| `Keyword arguments {'safety_checker': None} are not expected` | Passed to an SDXL pipeline | SDXL has no safety checker — use `load_kwargs`, not `SD15_KWARGS` |
| 401 / gated repo on download | Rare for these checkpoints | `from huggingface_hub import notebook_login; notebook_login()` |
| Very slow (minutes per image) | CPU runtime | `Runtime ▸ Change runtime type ▸ T4 GPU` |

---
# Part 9 — Responsible AI: what this method is, and is not

Worth stating plainly before anyone puts a restored photo in a family archive, a museum
catalogue or a case file.

1. **This is plausible reconstruction, not recovery.** Part 3.3 proved it: four seeds, four
   different faces in the torn band, all equally "valid". The model samples from what
   photographs *usually* look like. It has no access to what *this* photograph looked like.

2. **Invented detail is indistinguishable from recovered detail** in the output file. A
   viewer cannot tell which pixels were measured and which were imagined. If a restored image
   is going to be shared, the damaged original should be kept and the processing disclosed.

3. **Faces carry demographic risk.** Diffusion models inherit their training distribution.
   Restoring a face at high `strength` can drift skin tone, features and apparent age toward
   the dataset average. For portraits, keep `strength` low, and treat any facial change as a
   defect rather than an improvement.

4. **Never present a generated restoration as evidence.** Legal, medical, journalistic and
   historical contexts need provenance, and generative restoration destroys it.

5. **Open weights shift the accountability, they do not remove it.** Running locally means
   no photo leaves your machine — a real privacy gain over the hosted API. It also means no
   provider-side content policy is checking what you produce. That responsibility is now
   entirely yours.

---

## Deliberately out of scope

Noted explicitly so the boundary of this notebook is clear, and so these are available as
extension topics:

* **Face-specialised restoration** (GFPGAN, CodeFormer, RestoreFormer) — materially better on
  portraits than generic img2img, but a separate model family with its own failure modes.
* **ControlNet-guided restoration** (tile / lineart) — the standard way to hold structure
  fixed while raising `strength`; the honest fix for the strength trade-off in Part 3.1.
* **Diffusion super-resolution** (`stable-diffusion-x4-upscaler`, Real-ESRGAN) — we use an
  SDXL polish pass instead, which sharpens but does not truly upscale.
* **LoRA / DreamBooth fine-tuning** on a restoration dataset.
* **Perceptual metrics** (LPIPS, FID) — referenced in Part 6, not implemented; both need
  extra weights and FID needs a dataset, not a single image pair.
* **Scheduler comparison** (`DPMSolverMultistep`, `Euler a`, …) — left at each pipeline's
  default to keep the variable count teachable.
* **Automatic damage detection** — our tear mask is known by construction; real archival work
  needs a segmentation model or a hand-drawn mask.

---

## References

* [🤗 Diffusers — Image-to-image](https://huggingface.co/docs/diffusers/en/using-diffusers/img2img) *(the notebook this one is modelled on)*
* [🤗 Diffusers — Inpainting](https://huggingface.co/docs/diffusers/en/using-diffusers/inpaint)
* [🤗 Diffusers — AutoPipeline](https://huggingface.co/docs/diffusers/en/tutorials/autopipeline)
* [Stable Diffusion v1.5](https://huggingface.co/stable-diffusion-v1-5/stable-diffusion-v1-5) · [SD 1.5 Inpainting](https://huggingface.co/stable-diffusion-v1-5/stable-diffusion-inpainting) · [SDXL Refiner 1.0](https://huggingface.co/stabilityai/stable-diffusion-xl-refiner-1.0)
* [OpenAI Image generation guide](https://platform.openai.com/docs/guides/image-generation#edit-images) *(the approach this notebook replaces)*